# Hierarchical Graph Generator: Multi-Level ZINC Generation

One Edge Generator creates the top-level interpretation graph. Two conditional stages then generate the base molecule.

In [ ]:
from pathlib import Path
import runpy

roots = (Path.cwd(), *Path.cwd().parents)
candidates = [
    root / relative
    for root in roots
    for relative in (
        "notebooks/_bootstrap.py",
        "repos/abstractgraph-generative/notebooks/_bootstrap.py",
    )
]
bootstrap_path = next((path for path in candidates if path.is_file()), None)
if bootstrap_path is None:
    raise FileNotFoundError("Could not locate notebooks/_bootstrap.py")
runpy.run_path(str(bootstrap_path))

In [ ]:
from typing import cast

from sklearn.ensemble import RandomForestClassifier  # type: ignore[reportMissingModuleSource]

from abstractgraph.graphs import AbstractGraph
from abstractgraph.operators import (
    add,
    compose,
    cycle,
    intersection_edges,
    low_cut_partition,
    name,
    neighborhood,
    tree,
)
from abstractgraph.vectorize import AbstractGraphTransformer
from nsppk import NSPPK
from abstractgraph_graphicalizer.chem import ZINCLoader, draw_molecules
from abstractgraph_ml.estimators import GraphEstimator
from abstractgraph_ml.feasibility import (
    FeasibilityEstimator,
    FeasibilityEstimatorFeatureCannotExist,
)
from abstractgraph_generative.conditional import ConditionalAutoregressiveGenerator
from abstractgraph_generative.edge_generator import EdgeGenerator
from abstractgraph_generative.graph_generator import GraphGenerator

In [ ]:
graphs, _ = ZINCLoader(on_error="skip").load(
    "zinc_250k",
    limit=300,
    min_node_count=30,
    max_node_count=50,
)
print(f"Loaded {len(graphs)} molecules")

In [ ]:
def one_hop_neighborhood(abstract_graph: AbstractGraph) -> AbstractGraph:
    return cast(AbstractGraph, neighborhood(abstract_graph, radius=1))


# Bottom-up: molecules -> cycle/tree graph -> coarser cut graph.
base_decomposition = compose(
    intersection_edges(),
    add(compose(name("cycle"), cycle()), compose(name("tree"), tree())),
)
coarse_decomposition = compose(
    intersection_edges(),
    compose(name("cut"), low_cut_partition(
        max_part_size=4,
        min_part_size=3,
        target_max_boundary_nodes=2,
        target_max_cut_edges=3,
        min_overlap_nodes=1,
        seed=7,
    )),
)

conditionals = [
    ConditionalAutoregressiveGenerator(decomposition_function=base_decomposition, nbits=14),
    ConditionalAutoregressiveGenerator(decomposition_function=coarse_decomposition, nbits=14),
]
feasibility = FeasibilityEstimator([
    FeasibilityEstimatorFeatureCannotExist(
        decomposition_function=one_hop_neighborhood,
        nbits=19,
        parallel=False,
        n_jobs=1,
    )
])
vectorizer = cast(
    AbstractGraphTransformer,
    NSPPK(radius=1, distance=4, connector=0, nbits=14, dense=True),
)
edge_generator = EdgeGenerator(
    feasibility_estimator=feasibility,
    graph_estimator=GraphEstimator(
        transformer=vectorizer,
        estimator=RandomForestClassifier(
            n_estimators=80,
            class_weight="balanced_subsample",
            random_state=0,
            n_jobs=1,
        ),
    ),
    n_negative_per_positive=1,
    n_replicates=1,
    beam_size=3,
    max_restarts=1,
    fit_n_jobs=1,
    seed=0,
)
generator = GraphGenerator(edge_generator=edge_generator, conditional_generators=conditionals)

In [ ]:
generator.store(graphs)

In [ ]:
samples = generator.sample(
    n_samples=1,
    n_interpretation_neighbors=20,
    n_conditional_neighbors=[20, 20],
    random_state=0,
)
print(f"Generated {len(samples)} molecules")
if samples:
    draw_molecules(samples, n_graphs_per_line=len(samples))